In [55]:
import kagglehub
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

In [56]:
# ============== Preparing dataframes ==============
path = kagglehub.competition_download("titanic")
train_df = pd.read_csv(f"{path}/train.csv")
test_df = pd.read_csv(f"{path}/test.csv")

In [57]:
# ============== Processing ==============
# Train
train = train_df.copy()
train = train.drop(['Name', 'Ticket', 'PassengerId'], axis=1)
train['Sex'] = train['Sex'].map({'male': 0, 'female': 1})
mean_age = train['Age'].mean()
train['Age'] = train['Age'].fillna(mean_age).round()
train['Cabin'] = train['Cabin'].notnull().astype(int)
train['Embarked'] = train['Embarked'].fillna('S').map({'S': 2, 'C': 1, 'Q': 0})
train['FamilySize'] = train['SibSp'] + train['Parch'] + 1
train['IsAlone'] = (train['FamilySize'] == 1).astype(int)
train['Fare_log'] = np.log1p(train['Fare'])

y = train['Survived'].values
X = train.drop('Survived', axis=1).values

# Test
passenger_ids = test_df['PassengerId'].copy()
test = test_df.drop(['Name', 'Ticket', 'PassengerId'], axis=1)
test['Sex'] = test['Sex'].map({'male': 0, 'female': 1})
test['Age'] = test['Age'].fillna(mean_age).round()
test['Cabin'] = test['Cabin'].notnull().astype(int)
test['Embarked'] = test['Embarked'].fillna('S').map({'S': 2, 'C': 1, 'Q': 0})
median_fare = train['Fare'].median()
test['Fare'] = test['Fare'].fillna(median_fare)
test['Fare_log'] = np.log1p(test['Fare'])
test['FamilySize'] = test['SibSp'] + test['Parch'] + 1
test['IsAlone'] = (test['FamilySize'] == 1).astype(int)

feature_cols = train.drop('Survived', axis=1).columns
test = test[feature_cols]

X_test = test.values

In [58]:
# ============== XGBoost ==============
xgb = XGBClassifier(
    n_estimators=200,
    max_depth=3,       
    learning_rate=0.05,
    subsample=0.8,   
    colsample_bytree=0.8,
    reg_lambda=2,
    random_state=42
)
xgb.fit(X, y)

print("XGBoost на валидации:")
print(f"Validation accuracy: {xgb.score(X, y):.3f}")
predicts = xgb.predict(X)
print(f"precision: {precision_score(y, predicts):.3f}")
print(f"recall: {recall_score(y, predicts):.3f}")
print(f"F1-score: {f1_score(y, predicts):.3f}")

XGBoost на валидации:
Validation accuracy: 0.882
precision: 0.891
recall: 0.789
F1-score: 0.837


In [59]:
# ============== Final predictions ==============
final_predictions = xgb.predict(X_test)

In [60]:
# ============== Final csv ==============
submission = pd.DataFrame({
    'PassengerId': passenger_ids,
    'Survived': final_predictions.astype(int)
})

submission.to_csv('../../submissions/titanic/titanic-submission-v5.csv', index=False)
print(submission.head())

# Score: 0.77033

   PassengerId  Survived
0          892         0
1          893         0
2          894         0
3          895         0
4          896         0
